# สำรวจ response ของ Air4Thai (สัปดาห์ที่ 3)

ดูโครงสร้าง JSON จาก `getNewAQI_JSON.php` เพื่อจับคู่ key กับคอลัมน์ในตารางของเรา ก่อนเขียน `src/fetch_air4thai.py`

In [1]:
import json
from collections import Counter
from pprint import pprint

import requests

try:
    import truststore
    truststore.inject_into_ssl()
except ImportError:
    pass

URL = "http://air4thai.pcd.go.th/services/getNewAQI_JSON.php"
HEADERS = {"User-Agent": "pm25-explorer-0277/0.1 (student project; teerapon.po.67@ubu.ac.th)"}

try:
    r = requests.get(URL, headers=HEADERS, timeout=(10, 60))
    print("status:", r.status_code, "| content-type:", r.headers.get("Content-Type"), "| bytes:", len(r.content))
    data = r.json()
    source = "live API"
except Exception as e:
    print("ดึงสดไม่สำเร็จ:", type(e).__name__, "→ ใช้ ../data/sample/air4thai_sample.json")
    with open("../data/sample/air4thai_sample.json", encoding="utf-8") as f:
        data = json.load(f)
    source = "sample file"
print("แหล่งข้อมูล:", source)

status: 200 | content-type: application/json; charset=UTF-8 | bytes: 140118
แหล่งข้อมูล: live API


In [2]:
print("top-level:", type(data).__name__, list(data.keys()))
stations = data["stations"]
print("stations:", type(stations).__name__, len(stations), "สถานี")
pprint(stations[0])

top-level: dict ['stations']
stations: list 180 สถานี
{'AQILast': {'AQI': {'aqi': '21', 'color_id': '1', 'param': 'O3'},
             'CO': {'aqi': '4', 'color_id': '1', 'value': '0.70'},
             'NO2': {'aqi': '0', 'color_id': '1', 'value': '0'},
             'O3': {'aqi': '21', 'color_id': '1', 'value': '29'},
             'PM10': {'aqi': '-1', 'color_id': '0', 'value': '-1'},
             'PM25': {'aqi': '11', 'color_id': '1', 'value': '6.3'},
             'SO2': {'aqi': '1', 'color_id': '1', 'value': '2'},
             'date': '2026-11-25',
             'time': '15:00'},
 'areaEN': 'Pak Nam Subdistrict, Mueang District, Krabi',
 'areaTH': 'ต.ปากน้ำ อ.เมือง, กระบี่',
 'forecast': [],
 'lat': '8.0506237',
 'long': '98.9180489',
 'nameEN': 'Thara Public Park',
 'nameTH': 'สวนสาธารณะธารา',
 'stationID': '119t',
 'stationType': 'GROUND'}


### ทุกสถานีมี key เหมือนกันไหม

In [3]:
# นับรูปแบบ key ที่พบบ่อยที่สุด ทั้งระดับสถานีและใน AQILast
print(Counter(tuple(sorted(s.keys())) for s in stations).most_common(3))
print(Counter(tuple(sorted(s.get("AQILast", {}).keys())) for s in stations).most_common(3))

[(('AQILast', 'areaEN', 'areaTH', 'forecast', 'lat', 'long', 'nameEN', 'nameTH', 'stationID', 'stationType'), 180)]
[(('AQI', 'CO', 'NO2', 'O3', 'PM10', 'PM25', 'SO2', 'date', 'time'), 180)]


### ค่าหายแทนด้วยอะไร

In [4]:
for p in ["PM25", "PM10", "O3", "CO", "NO2", "SO2"]:
    vals = [s["AQILast"].get(p, {}).get("value") for s in stations]
    print(f"{p:5} ชนิด={dict(Counter(type(v).__name__ for v in vals))} "
          f"'-1'={sum(v == '-1' for v in vals):3} ว่าง/None={sum(v in ('', None) for v in vals)}")
print("ชนิดของ lat:", Counter(type(s["lat"]).__name__ for s in stations))
print("stationType:", Counter(s["stationType"] for s in stations))
print("วันที่ของค่าล่าสุด:", Counter(s["AQILast"]["date"] for s in stations).most_common(4))

PM25  ชนิด={'str': 180} '-1'=  1 ว่าง/None=0
PM10  ชนิด={'str': 180} '-1'=114 ว่าง/None=0
O3    ชนิด={'str': 180} '-1'=151 ว่าง/None=0
CO    ชนิด={'str': 180} '-1'=150 ว่าง/None=0
NO2   ชนิด={'str': 180} '-1'=147 ว่าง/None=0
SO2   ชนิด={'str': 180} '-1'=149 ว่าง/None=0
ชนิดของ lat: Counter({'str': 180})
stationType: Counter({'GROUND': 104, 'BKK': 68, 'MOBILE': 8})
วันที่ของค่าล่าสุด: [('2026-11-25', 175), ('2026-10-21', 1), ('2026-10-09', 1), ('2026-11-11', 1)]


### ตารางจับคู่ key

ตัวอย่างค่าจากสถานี `119t` (ข้อมูลจริงจาก Air4Thai) ถ้ารันสดแล้วค่าต่างไป ให้แก้คอลัมน์ตัวอย่างตามผลจริงของ cell ด้านบน

| คอลัมน์ในตารางของเรา | path ของ key ใน JSON | ชนิดใน JSON | ตัวอย่างค่า |
|---|---|---|---|
| `station_id` | `stations[].stationID` | string | `"119t"` |
| `name_th` | `stations[].nameTH` | string | `"สวนสาธารณะธารา"` |
| `name_en` | `stations[].nameEN` | string | `"Thara Public Park"` |
| `province` | อยู่ท้าย `stations[].areaTH` หลังเครื่องหมาย `,` | string | `"ต.ปากน้ำ อ.เมือง, กระบี่"` → `กระบี่` |
| `lat` | `stations[].lat` | string (ต้องแปลงเป็นตัวเลข) | `"8.0506237"` |
| `lon` | `stations[].long` | string (ต้องแปลงเป็นตัวเลข) | `"98.9180489"` |
| `station_type` | `stations[].stationType` | string | `"GROUND"` (มี `BKK`, `MOBILE` ด้วย) |
| วันที่วัด | `stations[].AQILast.date` | string | `"2026-11-18"` |
| เวลาที่วัด | `stations[].AQILast.time` | string | `"00:00"` |
| `pm25` | `stations[].AQILast.PM25.value` (ไม่ใช่ `.aqi`) | string | `"6.3"` |
| `pm10` | `stations[].AQILast.PM10.value` | string | `"-1"` (ไม่มีค่า) |
| `o3` | `stations[].AQILast.O3.value` | string | `"13"` |
| `co` | `stations[].AQILast.CO.value` | string | `"0.40"` |
| `no2` | `stations[].AQILast.NO2.value` | string | `"3"` |
| `so2` | `stations[].AQILast.SO2.value` | string | `"2"` |

**ค่าหายแทนด้วยอะไร:** ใช้ข้อความ `"-1"` ทั้งใน `value` และ `aqi` ไม่ใช่ `null` หรือช่องว่าง และตัวเลขทุกตัวเก็บเป็น string ต้องแปลงด้วย `pd.to_numeric` แล้วเปลี่ยน `-1` เป็นค่าหายก่อนคำนวณ

**ทุกสถานีมี key เหมือนกันไหม:** ในข้อมูลที่ตรวจ ทุกสถานีมีชุด key เหมือนกัน ทั้งระดับสถานีและใน `AQILast` แต่สถานีที่ไม่ได้วัดสารบางตัวยังมี key นั้นอยู่ โดยค่าเป็น `"-1"` และบางสถานีมีวันที่ค่าล่าสุดเก่ากว่าสถานีอื่น

### ตรวจ robots.txt

In [5]:
from urllib.robotparser import RobotFileParser

rp = RobotFileParser("http://air4thai.pcd.go.th/robots.txt")
rp.read()
rp.can_fetch(HEADERS["User-Agent"], URL)

True

ผลจาก cell ด้านบน (`True` = อนุญาต) นำไปเขียนในหัวข้อ "แหล่งข้อมูล" ของ `README.md`